# 00 — Tensor Basics

The goal of this lesson is to develop intuition for tensor shapes
and the operations used later in attention.

In [13]:
import torch

In [14]:
# Basic construction
scalar = torch.tensor(7)
vector = torch.tensor([7, 8, 9])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])

print("scalar:", scalar.ndim, scalar.shape, scalar.dtype)
print("vector:", vector.ndim, vector.shape, vector.dtype)
print("matrix:", matrix.ndim, matrix.shape, matrix.dtype)


scalar: 0 torch.Size([]) torch.int64
vector: 1 torch.Size([3]) torch.int64
matrix: 2 torch.Size([2, 3]) torch.int64


In [15]:
# unsqueeze: add 1 dim along with unsqueeze(x)
vector = torch.tensor([7, 8, 9])

row = vector.unsqueeze(0)
col = vector.unsqueeze(1)

print("vector: ", vector.shape)
print("row: ", row.shape)
print("col: ", col.shape)


vector:  torch.Size([3])
row:  torch.Size([1, 3])
col:  torch.Size([3, 1])


In [16]:
# indexing
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(matrix[0])  # =>  [1, 2, 3]
print(matrix[1])  # => [4, 5, 6]
print(matrix[0, 2])  # => 3

tensor([1, 2, 3])
tensor([4, 5, 6])
tensor(3)


In [17]:
# squeeze: reduce 1 dim along with squeeze(x)
row = torch.tensor([[7, 8, 9]])
squeezed = row.squeeze(0)  # => tensor([7,8,9])
print("row: ", row.shape)
print("shaped: ", squeezed.shape)

col = torch.tensor(
    [
        [7],
        [8],
        [9],
    ]
)

squeezed_col = col.squeeze(1)  # => tensor([7,8,9])


row:  torch.Size([1, 3])
shaped:  torch.Size([3])


In [18]:
# reshape: reshape the tensor
x = torch.tensor([1, 2, 3, 4, 5, 6])
a = x.reshape(2, 3)
b = x.reshape(3, 2)

In [19]:
# transpose: exchange the dim of the tensor - (2,3) => (3,2) with transpose(0,1)
matrix = torch.tensor(
    [
        [1, 2, 3],
        [4, 5, 6],
    ]
)

transposed = matrix.transpose(0, 1)
# transposed = [
#   [1, 4]
#   [2, 5]
#   [3, 6]
# ]


In [20]:
# matrix multification
a = torch.tensor(
    [
        [1, 2, 3],
        [4, 5, 6],
    ]
)

b = torch.tensor(
    [
        [7, 8],
        [9, 10],
        [11, 12],
    ]
)
c = a @ b  # matrix mul
print("c:", c)
print(c.shape)
print(c[0, 0])
print(c[0, 1])
print(c[1, 0])


c: tensor([[ 58,  64],
        [139, 154]])
torch.Size([2, 2])
tensor(58)
tensor(64)
tensor(139)


In [21]:
# QKV attention

q = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

k = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

k_t = k.transpose(0, 1)
scores = q @ k_t
print("scores:", scores)


d_k = q.shape[-1]
scaled_scores = scores / (d_k**0.5)

print("scores: \n", scores)
print("scaled scores: \n", scaled_scores)

scores: tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])
scores: 
 tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])
scaled scores: 
 tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])


In [22]:
# softmax
def softmax_naive(x):
    """
    dim=-1:
        reduce over the last dimension

    keepdim=True:
        keep the reduced dimension with size 1
        instead of removing it
    """
    exp_x = torch.exp(x)
    row_sums = exp_x.sum(dim=-1, keepdim=True)
    return exp_x / row_sums


manual_weights = softmax_naive(scaled_scores)

print(manual_weights)
print(manual_weights.sum(dim=-1))

# if the naive softmax works as builtin `softmax` func
attn_weights = torch.softmax(scaled_scores, dim=-1)
print(attn_weights)
print(attn_weights == manual_weights)


tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
tensor([1., 1., 1.])
tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
tensor([[False,  True, False],
        [ True, False, False],
        [ True,  True,  True]])


In [23]:
# check the diff, and remember to use `torch.allclose() to comapre the float tensor`
diff = (attn_weights - manual_weights).abs()

print(diff)
print("max difference:", diff.max())
print(torch.allclose(attn_weights, manual_weights))


# test softmax naive
large = torch.tensor([1000.0, 1001.0, 1002.0])

print(torch.exp(large))
print(softmax_naive(large))

tensor([[2.9802e-08, 0.0000e+00, 2.9802e-08],
        [0.0000e+00, 2.9802e-08, 2.9802e-08],
        [0.0000e+00, 0.0000e+00, 0.0000e+00]])
max difference: tensor(2.9802e-08)
True
tensor([inf, inf, inf])
tensor([nan, nan, nan])


In [24]:
def softmax_stable(x):
    max_x = x.max(
        dim=-1, keepdim=True
    ).values  # .values => get values, here [0] the same.
    shifted_x = x - max_x

    exp_x = torch.exp(shifted_x)
    row_sums = exp_x.sum(dim=-1, keepdim=True)

    return exp_x / row_sums


large = torch.tensor([1000.0, 1001.0, 1002.0])

print("test softmax naive")
print("naive:", softmax_naive(large))
print("stable:", softmax_stable(large))
print("torch:", torch.softmax(large, dim=-1))

print(
    torch.allclose(
        softmax_stable(large),
        torch.softmax(large, dim=-1),
    )
)

test softmax naive
naive: tensor([nan, nan, nan])
stable: tensor([0.0900, 0.2447, 0.6652])
torch: tensor([0.0900, 0.2447, 0.6652])
True


## 12. Broadcasting

Broadcasting allows tensors with compatible shapes to participate in
element-wise operations without explicitly copying the smaller tensor.

In [25]:
x = torch.tensor(
    [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
    ]
)

bias = torch.tensor([10, 20, 30, 40])

print(x + bias)


tensor([[11, 22, 33, 44],
        [15, 26, 37, 48],
        [19, 30, 41, 52]])


In [26]:
column = torch.tensor(
    [
        [100],
        [200],
        [300],
    ]
)
print("column shape:", column.shape)
print("bias shape:", bias.shape)

result = column + bias

print(result)
print(result.shape)

column shape: torch.Size([3, 1])
bias shape: torch.Size([4])
tensor([[110, 120, 130, 140],
        [210, 220, 230, 240],
        [310, 320, 330, 340]])
torch.Size([3, 4])


## 13. Element-wise vs Matrix Multiplication

In [27]:
a = torch.tensor(
    [
        [1, 2],
        [3, 4],
    ]
)

b = torch.tensor(
    [
        [5, 6],
        [7, 8],
    ]
)

In [29]:
elementwise = a * b
matmul = a @ b

print("a * b: \n", elementwise)
print("a @ b: \n", matmul)


a * b: 
 tensor([[ 5, 12],
        [21, 32]])
a @ b: 
 tensor([[19, 22],
        [43, 50]])


## Device：CPU -> GPU

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [32]:
x = torch.tensor([1.0, 2.0, 3.0])
print("before: ", x.device)

x = x.to(device)
print("after", x.device)

before:  cpu
after cuda:0


In [33]:
# the tensor must be in the **same** device
cpu_x = torch.tensor([1.0, 2.0])
gpu_y = torch.tensor([3.0, 4.0]).to("cuda")

cpu_x + gpu_y

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## 16. Autograd

In [34]:
x = torch.tensor(
    [2.0, 3.0],
    requires_grad=True,
)
y = (x**2).sum()

print("x:", x)
print("y:", y)

y.backward()
print(x.grad)

x: tensor([2., 3.], requires_grad=True)
y: tensor(13., grad_fn=<SumBackward0>)
tensor([4., 6.])


In [ ]:
# clean grad
x.grad.zero_()
y = (x**2).sum()
y.backward()

print(x.grad)


tensor([4., 6.])


## Takeaways

- A tensor dimension is an indexing axis, not inherently a row or column.
- Tensor shapes describe the size of each axis.
- Broadcasting aligns dimensions from right to left.
- `*` is element-wise multiplication, while `@` performs matrix multiplication.
- Reductions such as `sum` can remove a dimension unless `keepdim=True`.
- Stable softmax subtracts the maximum value before exponentiation.
- Tensors participating in an operation need compatible devices.
- Autograd records operations and computes gradients through backpropagation.